In [3]:
import cv2
import os
from ultralytics import YOLO
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt

# configuration
STREAM_FILE = "live_footage.mp4"
OUTPUT_DIR = "../data/outputs/outputsv5"
GRAPH_DIR = "../visualization/graphs"
DATA_DIR = "../data/new_data"
FPS = 10  # fps of output video
SAMPLE_FRAMES = 120  # fps that are actually used

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# yolo model
model = YOLO("yolov8n-seg.pt")

# opening video
cap = cv2.VideoCapture(STREAM_FILE)
if not cap.isOpened():
    raise Exception(f"Cannot open video: {STREAM_FILE}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_duration_sec = total_frames / cap.get(cv2.CAP_PROP_FPS)

# framing sampling interval
sample_interval = max(1, total_frames // SAMPLE_FRAMES)

output_path = os.path.join(OUTPUT_DIR, "processed_video.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, FPS, (frame_width, frame_height))

people_counts = []
timestamps = []

# processing frame by frame
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # only processing sample frames
    if frame_idx % sample_interval != 0:
        frame_idx += 1
        continue

    results = model(frame)[0]

    people_count = 0
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)

    if results.masks is not None:
        for r in results.masks.xy:
            people_count += 1
            poly = np.array(r, dtype=np.int32)
            cv2.fillPoly(mask, [poly], 255)

    # blurring of the individuals
    if people_count > 0:
        blurred = cv2.GaussianBlur(frame, (51, 51), 0)
        frame = np.where(mask[:, :, None] == 255, blurred, frame)

    out.write(frame)

    # needed data for forecasting
    people_counts.append(people_count)
    timestamps.append(frame_idx / cap.get(cv2.CAP_PROP_FPS))

    frame_idx += 1
    print(f"Frame {frame_idx}/{total_frames} | People detected: {people_count}")

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Video saved in: {output_path}")

# preparing data for prophet forecasting
df = pd.DataFrame({
    "ds": pd.to_datetime(timestamps, unit='s'),
    "y": people_counts
})

# forecast model
model_prophet = Prophet(daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=False)
model_prophet.fit(df)

# forecasting 6 hours into the future
future = model_prophet.make_future_dataframe(periods=6*3600, freq='s')
forecast = model_prophet.predict(future)

# plotting and saving of the graph
plt.figure(figsize=(16,8))

# x-axis for actual counts in minutes
x_actual = np.array(timestamps) / 60
plt.plot(x_actual, people_counts, label='Actual Count', color='blue', linewidth=2)

# starting of forecast = last timestamp of actual count
forecast_start_sec = timestamps[-1]

# only taking 6 hours into the future
forecast_6h = forecast[forecast['ds'] > pd.to_datetime(forecast_start_sec, unit='s')]
forecast_6h = forecast_6h[forecast_6h['ds'] <= pd.to_datetime(forecast_start_sec, unit='s') + pd.Timedelta(hours=6)]

# x-axis for forecast 
x_forecast = (forecast_6h['ds'] - pd.to_datetime(forecast_start_sec, unit='s')).dt.total_seconds() / 3600
plt.plot(x_forecast, forecast_6h['yhat'], label='Forecast (next 6h)',
         color='orange', linestyle='--', linewidth=2)

plt.xlabel('Time (minutes / hours)')
plt.ylabel('People count')
plt.title('People count forecast')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAPH_DIR, 'count_forecast_v5.png'))
plt.close()
print(f"Graph saved in: {GRAPH_DIR}")

df.to_csv(os.path.join(DATA_DIR, 'people_countv5.csv'), index=False)

# forecast
forecast[['ds','yhat']].to_csv(os.path.join(DATA_DIR, 'people_forecastv5.csv'), index=False)
print(f"Csv saved in: {DATA_DIR}")


0: 288x640 (no detections), 32.4ms
Speed: 0.9ms preprocess, 32.4ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 640)
Frame 1/17616 | People detected: 0

0: 288x640 (no detections), 26.3ms
Speed: 0.6ms preprocess, 26.3ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 640)
Frame 147/17616 | People detected: 0

0: 288x640 (no detections), 27.8ms
Speed: 0.6ms preprocess, 27.8ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 640)
Frame 293/17616 | People detected: 0

0: 288x640 1 person, 29.4ms
Speed: 0.7ms preprocess, 29.4ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)
Frame 439/17616 | People detected: 1

0: 288x640 (no detections), 28.8ms
Speed: 0.6ms preprocess, 28.8ms inference, 0.3ms postprocess per image at shape (1, 3, 288, 640)
Frame 585/17616 | People detected: 0

0: 288x640 2 persons, 29.3ms
Speed: 0.7ms preprocess, 29.3ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 640)
Frame 731/17616 | People detecte

14:34:59 - cmdstanpy - INFO - Chain [1] start processing


Frame 17521/17616 | People detected: 2
Video saved in: ../data/outputs/outputsv5/processed_video.mp4


14:34:59 - cmdstanpy - INFO - Chain [1] done processing


Graph saved in: ../visualization/graphs
Csv saved in: ../data/new_data
